In [0]:
from TornAPI.Torn import Faction
from pyspark.sql.functions import lit

import datetime as dt

faction_api = Faction(dbutils.secrets.get("Personal", "TornAPI"))

In [0]:

if spark.catalog.tableExists("torn.faction.balances"):
    sp_in = spark.read.table("torn.faction.balances")
    latest_date = sp_in.select("date").agg({"date": "max"}).collect()
    latest_date = latest_date[0].asDict()["max(date)"]
else:
    latest_date = dt.date.today() + dt.timedelta(days=-1)

In [0]:
if dt.date.today() > latest_date:
    balance_data = faction_api.get_balance()

    sp_balance_data = spark.createDataFrame(balance_data["balance"]["members"])

    sp_balance_data = sp_balance_data.withColumn("date", lit(dt.date.today()))
    
    sp_balance_data.write.format("delta").mode("append").saveAsTable("torn.faction.balances")